# RAG : comparaison avec et sans recherche de passages

1. **Préparation** (commun aux deux approches) : imports, données, LLM, métriques
2. **Non-RAG** (QA génératif fermé) : le LLM répond seul aux 50 questions
3. **RAG** (QA génératif ouvert) : recherche de passages Wikipedia avec FAISS, puis réponse du même LLM
4. **Comparaison RAG vs Non-RAG** : analyse quantitative et qualitative

## 1. Préparation

Éléments communs au Non-RAG et au RAG, définis une seule fois pour que les deux approches
soient comparées avec le même LLM et les mêmes métriques.

### 1.1 Imports et configuration

In [1]:
import json
import random

import faiss
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Dossiers utilisés dans cette section
DOSSIER_DONNEES = Path("data")
DOSSIER_INDEX = DOSSIER_DONNEES / "index_faiss"
DOSSIER_RESULTATS = Path("results")

DOSSIER_INDEX.mkdir(parents=True, exist_ok=True)
DOSSIER_RESULTATS.mkdir(parents=True, exist_ok=True)

### 1.2 Données

Chargement des 50 questions et des passages Wikipedia.

In [2]:
with open(DOSSIER_DONNEES / "train_texts.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

with open(DOSSIER_DONNEES / "test_questions.json", "r", encoding="utf-8") as f:
    questions = json.load(f)

print("Nombre de passages :", len(documents))
print("Nombre de questions :", len(questions))

Nombre de passages : 747
Nombre de questions : 50


In [3]:
# Combinaison du titre, de la section et du texte
passages = [
    f"{doc['title']}\n{doc['section']}\n{doc['text']}"
    for doc in documents
]

print("\nExemple de passage :\n")
print(passages[0])


Exemple de passage :

GPT-3
Introduction
Generative Pre-trained Transformer 3 (GPT-3) was a large language model released by OpenAI in 2020 as part of the company's GPT series of models. Like its predecessor, GPT-2, it is a decoder-only transformer model of deep neural network, which supersedes recurrence and convolution-based architectures with a technique known as "attention". This attention mechanism allows the model to focus selectively on segments of input text it predicts to be most relevant. GPT-3 has 175 billion parameters, each with 16-bit precision, requiring 350GB of storage since each parameter occupies 2 bytes. It has a context window size of 2,048 tokens, and has demonstrated strong "zero-shot" and "few-shot" learning abilities on many tasks.


### 1.3 LLM

Chargement du modèle et fonction pour générer une réponse.


### 1.4 Métriques

F1 sur les jetons, plus une ou deux autres métriques.


In [4]:
import string
import re
from collections import Counter

def remove_articles(text):
    return re.sub(r'\b(a|an|the)\b', ' ', text)

def white_space_fix(text):
    return ' '.join(text.split())

def remove_punc(text):
    exclude = set(string.punctuation)
    return ''.join(ch for ch in text if ch not in exclude)

def lower(text):
    return text.lower()

def normalize_answer(s):
    """Mettre en minuscule et retirer la ponctuation, des déterminants and les espaces."""
    return white_space_fix(remove_articles(remove_punc(lower(s))))

In [5]:
def evaluate_f1(ground_truth, prediction):
    """Normalise les 2 textes, trouve ce qu'il y a en commun et estime précision, rappel et F1."""
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0, 0.0, 0.0
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return precision, recall, f1

## 2. Non-RAG (QA génératif fermé)


## 3. RAG (QA génératif ouvert)

Pipeline : les passages Wikipedia sont indexés avec FAISS (**retriever**), puis les k passages
les plus proches de la question sont donnés au même LLM (**reader / generator**).

- 3.1 Indexation : découpage, encodeurs, index FAISS
- 3.2 Choix de l'encodeur : MiniLM vs E5
- 3.3 Choix de k
- 3.4 Génération : recherche des passages, prompt, 50 réponses et scores

### 3.1 Indexation

#### 3.1.1 Modèles d'embeddings

- **MiniLM** (`paraphrase-multilingual-MiniLM-L12-v2`) : petit modèle, limite fixée à 256 tokens.
- **E5** (`multilingual-e5-large-instruct`) : modèle plus grand, entraîné pour la recherche, limite de 512 tokens.
  Les requêtes reçoivent une instruction, les passages aucun préfixe.

In [6]:
# Charger les modèles d'embeddings une seule fois : MiniLM, E5

# Identifiants des modèles
MODELE_MINILM = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODELE_E5 = "intfloat/multilingual-e5-large-instruct"

# Charger MiniLM
modele_minilm = SentenceTransformer(MODELE_MINILM)
modele_minilm.max_seq_length = 256

# Charger E5
modele_e5 = SentenceTransformer(MODELE_E5)

# Regrouper les modèles
modeles = {
    "minilm": modele_minilm,
    "e5": modele_e5,
}

# Afficher les limites de chaque modèle
for nom, modele in modeles.items():
    print(f"{nom} : limite configurée de {modele.max_seq_length} tokens")


# Format des requêtes
# E5-instruct attend "Instruct: <tâche>\nQuery: <question>" pour les requêtes.
# Les passages sont encodés sans préfixe. MiniLM n'utilise aucun préfixe.
INSTRUCTION_E5 = "Given a question, retrieve Wikipedia passages that answer the question"

def formater_requete(nom, question):
    if nom == "e5":
        return f"Instruct: {INSTRUCTION_E5}\nQuery: {question}"
    return question

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

minilm : limite configurée de 256 tokens
e5 : limite configurée de 512 tokens


#### 3.1.2 Longueur des passages et découpage

On vérifie combien de passages dépassent la limite de chaque modèle. Seuls ces passages sont découpés
(taille = limite du modèle, chevauchement de 40 tokens), en répétant le titre et la section dans chaque morceau.

In [7]:
# Vérifier les longueurs des passages avant découpage

resultats_longueurs = []

for nom, modele in modeles.items():

    tokenizer = modele.tokenizer
    limite = modele.max_seq_length

    longueurs = [
        len(tokenizer.encode(
            texte,
            add_special_tokens=True,
            truncation=False,
            verbose=False
        ))
        for texte in passages
    ]

    resultats_longueurs.append({
        "Modèle": nom,
        "Nombre de passages": len(longueurs),
        "Limite tokens": limite,
        "Maximum observé": max(longueurs) if longueurs else 0,
        "Médiane": int(pd.Series(longueurs).median()) if longueurs else 0,
        "Passages dépassant la limite": sum(
            longueur > limite for longueur in longueurs
        )
    })

df_longueurs = pd.DataFrame(resultats_longueurs)

display(df_longueurs)

,Modèle,Nombre de passages,Limite tokens,Maximum observé,Médiane,Passages dépassant la limite
0,minilm,747,256,286,179,6
1,e5,747,512,286,179,0


In [8]:
def decouper_passages(documents, modele, taille=200, chevauchement=40):

    tokenizer = modele.tokenizer
    limite = modele.max_seq_length
    morceaux = []

    assert taille > 0
    assert 0 <= chevauchement < taille

    for doc in documents:

        # Informations conservées dans chaque morceau
        titre = doc["title"]
        section = doc["section"]
        texte = doc["text"]

        entete = f"{titre}\n{section}\n"

        # Tokens de l'en-tête
        nb_entete = len(tokenizer.encode(
            entete, add_special_tokens=False
        ))

        # Place disponible pour le texte
        nb_speciaux = tokenizer.num_special_tokens_to_add(pair=False)
        espace = min(taille, limite) - nb_entete - nb_speciaux

        if espace <= chevauchement:
            raise ValueError("En-tête trop long ou chevauchement trop grand")

        # Découpage du texte seulement
        tokens = tokenizer.encode(
            texte,
            add_special_tokens=False,
            truncation=False
        )

        pas = espace - chevauchement

        for numero, debut in enumerate(range(0, len(tokens), pas), start=1):

            fin = min(debut + espace, len(tokens))

            morceau_texte = tokenizer.decode(
                tokens[debut:fin],
                skip_special_tokens=True
            )

            morceau = entete + morceau_texte

            morceaux.append({
                "chunk_id": doc["chunk_id"],
                "numero_morceau": numero,
                "texte_embedding": morceau
            })

            if fin == len(tokens):
                break

    return pd.DataFrame(morceaux)

# Découpage MiniLM
morceaux_minilm = decouper_passages(
    documents,
    modele_minilm,
    taille=modele_minilm.max_seq_length,
    chevauchement=40
)

morceaux_e5 = decouper_passages(
    documents, 
    modele_e5,
    taille=modele_e5.max_seq_length,
    chevauchement=40
)

print("MiniLM :", len(morceaux_minilm), "morceaux")
print("E5 :", len(morceaux_e5), "morceaux")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (264 > 256). Running this sequence through the model will result in indexing errors


MiniLM : 753 morceaux
E5 : 747 morceaux


#### 3.1.3 Construction des index FAISS

Index exact `IndexFlatIP` sur des vecteurs normalisés : le produit scalaire correspond à la similarité cosinus.
Les `chunk_id` sont sauvegardés dans le même ordre que les vecteurs.

In [9]:
# Construction et sauvegarde des index FAISS (un par modèle)

morceaux_par_modele = {
    "minilm": morceaux_minilm,
    "e5": morceaux_e5,
}

for nom, modele in modeles.items():

    morceaux = morceaux_par_modele[nom]

    # Embeddings des passages (aucun préfixe pour les deux modèles)
    vecteurs = modele.encode(
        morceaux["texte_embedding"].tolist(),
        batch_size=32,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    ).astype("float32")

    # Index exact par produit scalaire (similarité cosinus sur vecteurs normalisés)
    index = faiss.IndexFlatIP(vecteurs.shape[1])
    index.add(vecteurs)

    # Sauvegarder l'index et les chunk_id dans le même ordre que les vecteurs
    faiss.write_index(index, str(DOSSIER_INDEX / f"{nom}.faiss"))
    with open(DOSSIER_INDEX / f"{nom}.json", "w", encoding="utf-8") as f:
        json.dump(morceaux["chunk_id"].tolist(), f, ensure_ascii=False)

    print(f"{nom} : {index.ntotal} vecteurs de dimension {vecteurs.shape[1]}")

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

minilm : 753 vecteurs de dimension 384


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

e5 : 747 vecteurs de dimension 1024


### 3.2 Choix de l'encodeur : MiniLM vs E5

Protocole :

- 5 questions tirées au hasard (`random.seed(42)`), les mêmes pour les deux modèles ;
- pour chaque question, les 10 premiers passages **distincts** retournés par chaque index ;
- annotation manuelle de la pertinence (1/0) dans `results/evaluation_<modèle>.xlsx` ;
- calcul de Precision@k, RR et AP par question, puis MRR@10 et MAP@10 par modèle.

In [10]:
# Choisir les mêmes 5 questions pour les deux modèles
random.seed(42)
questions_selectionnees = random.sample(
    list(enumerate(questions)), 5
)

# Retrouver les textes originaux grâce au chunk_id
documents_par_id = {
    doc["chunk_id"]: doc for doc in documents
}

for nom, modele in modeles.items():

    # Charger FAISS et les identifiants
    index = faiss.read_index(str(DOSSIER_INDEX / f"{nom}.faiss"))

    with open(DOSSIER_INDEX / f"{nom}.json", encoding="utf-8") as f:
        ids = json.load(f)

    assert len(ids) == index.ntotal

    # Retrouver les morceaux dans leur ordre d'indexation
    morceaux = morceaux_e5 if nom.lower() == "e5" else morceaux_minilm
    assert len(morceaux) == index.ntotal
    assert ids == morceaux["chunk_id"].tolist()

    resultats = []

    for indice_question, question in questions_selectionnees:

        texte_question = question["question"]

        # Format de requête propre à chaque modèle
        # (E5-instruct : instruction + question ; MiniLM : question brute)
        requete = formater_requete(nom, texte_question)

        # Embedding de la question
        vecteur = modele.encode(
            [requete],
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype("float32")

        # Rechercher davantage de morceaux pour compenser les doublons
        scores, positions = index.search(
            vecteur,
            min(20, index.ntotal)
        )

        # Éviter de récupérer deux fois le même passage original
        chunk_ids_vus = set()
        rang = 0

        for score, position in zip(scores[0], positions[0]):

            if position < 0:
                continue

            chunk_id = ids[int(position)]

            # Ignorer si ce chunk_id a déjà été retrouvé
            if chunk_id in chunk_ids_vus:
                continue

            chunk_ids_vus.add(chunk_id)

            doc = documents_par_id[chunk_id]
            morceau = morceaux.iloc[int(position)]

            # Rang parmi les passages distincts
            rang += 1

            resultats.append({
                "Question ID": indice_question + 1,
                "Question": texte_question,
                "Réponse attendue": question["answer"],
                "Rang": rang,
                "Position FAISS": int(position),
                "Chunk ID": chunk_id,
                "Score FAISS": float(score),
                "Titre": doc["title"],
                "Section": doc["section"],
                "Texte retrouvé": doc["text"],
                "Pertinent (1/0)": None
            })

            # Arrêter après 10 passages différents
            if rang == 10:
                break

    # Exporter vers Excel
    df_resultats = pd.DataFrame(resultats)

    fichier = DOSSIER_RESULTATS / f"evaluation_{nom}.xlsx"

    # Ne jamais écraser un fichier déjà annoté à la main
    if fichier.exists():
        print(f"{nom} : {fichier} existe déjà, non écrasé (annotations conservées)")
    else:
        df_resultats.to_excel(fichier, index=False)
        print(f"{nom} : {len(df_resultats)} résultats → {fichier}")

minilm : 50 résultats → results\evaluation_minilm.xlsx
e5 : 50 résultats → results\evaluation_e5.xlsx


In [13]:
k_valeurs = [2, 3, 4, 5, 10]
resultats = []

for nom in ["minilm", "e5"]:

    df = pd.read_excel(DOSSIER_RESULTATS / f"evaluation_{nom}.xlsx")

    for question_id, groupe in df.groupby("Question ID"):

        # Rangs des passages annotés pertinents (lus dans la colonne Rang)
        rangs_pertinents = np.sort(groupe.loc[groupe["Pertinent (1/0)"] == 1, "Rang"].to_numpy())

        # Precision@k : nombre de pertinents de rang <= k, divisé par k
        precisions = {f"Precision@{k}": (rangs_pertinents <= k).sum() / k for k in k_valeurs}

        # RR : inverse du rang du premier pertinent
        rr = 1 / rangs_pertinents[0] if len(rangs_pertinents) > 0 else 0.0

        # AP : moyenne des précisions aux rangs des pertinents
        ap = np.mean([i / r for i, r in enumerate(rangs_pertinents, start=1)]) if len(rangs_pertinents) > 0 else 0.0

        resultats.append({"Modèle": nom, "Question ID": question_id, **precisions, "AP@10": ap, "RR@10": rr})

# Métriques par question
df_questions = pd.DataFrame(resultats)
display(df_questions.round(4))

# Moyennes par modèle (MAP et MRR)
df_comparaison = (
    df_questions.drop(columns="Question ID")
    .groupby("Modèle").mean()
    .rename(columns={"AP@10": "MAP@10", "RR@10": "MRR@10"})
    .round(4)
)
display(df_comparaison)

# Export
with pd.ExcelWriter(DOSSIER_RESULTATS / "comparaison_metriques.xlsx") as writer:
    df_questions.round(4).to_excel(writer, sheet_name="Par question", index=False)
    df_comparaison.to_excel(writer, sheet_name="Comparaison modèles")

,Modèle,Question ID,Precision@2,Precision@3,Precision@4,Precision@5,Precision@10,AP@10,RR@10
0,minilm,2,0.0,0.0000,0.00,0.0,0.1,0.1111,0.1111
1,minilm,8,0.5,0.3333,0.25,0.2,0.2,0.3611,0.5000
2,minilm,18,0.0,0.3333,0.25,0.2,0.1,0.3333,0.3333
3,minilm,41,0.5,0.6667,0.50,0.4,0.3,0.6556,1.0000
4,minilm,48,0.5,0.3333,0.50,0.4,0.4,0.5799,1.0000
5,e5,2,1.0,0.6667,0.50,0.6,0.3,0.8667,1.0000
6,e5,8,1.0,0.6667,0.75,0.6,0.3,0.9167,1.0000
7,e5,18,0.5,0.3333,0.25,0.2,0.1,1.0000,1.0000
8,e5,41,0.5,0.3333,0.50,0.6,0.4,0.6679,1.0000
9,e5,48,1.0,0.6667,0.50,0.6,0.5,0.7962,1.0000


,Precision@2,Precision@3,Precision@4,Precision@5,Precision@10,MAP@10,MRR@10
Modèle,,,,,,,
e5,0.8,0.5333,0.5,0.52,0.32,0.8495,1.0000
minilm,0.3,0.3333,0.3,0.24,0.22,0.4082,0.5889


**Conclusion :** E5 est nettement meilleur que MiniLM sur toutes les métriques
(MAP@10 : 0,85 contre 0,41 ; MRR@10 : 1,00 contre 0,59 ; Precision@5 : 0,52 contre 0,24).
E5 a toujours un passage pertinent comme première réponse (MRR@10 = 1). 

**E5 est retenu pour le RAG.**

### 3.3 Choix de k

Pour E5 seulement, comment évolue les métriques selon k passages. 

In [14]:
# Choix de k pour E5 : précision et pertinents récupérés selon le nombre de passages
df = pd.read_excel(DOSSIER_RESULTATS / "evaluation_e5.xlsx")
rangs = df.loc[df["Pertinent (1/0)"] == 1, ["Question ID", "Rang"]]
total_pertinents = len(rangs)
nb_questions = df["Question ID"].nunique()

lignes = []

for k in range(1, 11):
    dans_top_k = rangs[rangs["Rang"] <= k]
    lignes.append({
        "k": k,
        "Precision@k": len(dans_top_k) / (k * nb_questions),
        "Pertinents récupérés": f"{len(dans_top_k)} / {total_pertinents}",
        "% pertinents récupérés": len(dans_top_k) / total_pertinents,
        "Questions avec ≥ 1 pertinent": dans_top_k["Question ID"].nunique(),
    })

df_choix_k = pd.DataFrame(lignes).round(2)
display(df_choix_k)

,k,Precision@k,Pertinents récupérés,% pertinents récupérés,Questions avec ≥ 1 pertinent
0,1,1.00,5 / 16,0.31,5
1,2,0.80,8 / 16,0.50,5
2,3,0.53,8 / 16,0.50,5
3,4,0.50,10 / 16,0.62,5
4,5,0.52,13 / 16,0.81,5
5,6,0.47,14 / 16,0.88,5
6,7,0.46,16 / 16,1.00,5
7,8,0.40,16 / 16,1.00,5
8,9,0.36,16 / 16,1.00,5
9,10,0.32,16 / 16,1.00,5


**Conclusion :** on retient **k = 5**. 

Avec 5 passages, on récupère 81 % des passages pertinents
(13 / 16) avec une précision de 0,52, et chaque question a au moins un passage pertinent.

Au-delà de k = 7, il n'y a plus de gain : tous les passages pertinents trouvés sont déjà dans le top 7.

### 3.4 Génération

#### 3.4.1 Recherche des passages

La fonction `rechercher(question)` retourne les `K_RAG` passages distincts les plus proches
de la question avec l'index E5.

In [15]:
# Configuration retenue pour le RAG
NOM_MODELE_RAG = "e5"
K_RAG = 5

modele_rag = modeles[NOM_MODELE_RAG]
index_rag = faiss.read_index(str(DOSSIER_INDEX / f"{NOM_MODELE_RAG}.faiss"))

with open(DOSSIER_INDEX / f"{NOM_MODELE_RAG}.json", encoding="utf-8") as f:
    ids_rag = json.load(f)

documents_par_id = {doc["chunk_id"]: doc for doc in documents}

def rechercher(question, k=K_RAG):
    """Retourne les k passages distincts les plus proches de la question."""

    vecteur = modele_rag.encode(
        [formater_requete(NOM_MODELE_RAG, question)],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    # Chercher un peu plus large pour pouvoir retirer les doublons
    scores, positions = index_rag.search(vecteur, min(3 * k, index_rag.ntotal))

    passages_trouves = []
    chunk_ids_vus = set()

    for score, position in zip(scores[0], positions[0]):

        if position < 0:
            continue

        chunk_id = ids_rag[int(position)]
        if chunk_id in chunk_ids_vus:
            continue
        chunk_ids_vus.add(chunk_id)

        doc = documents_par_id[chunk_id]
        passages_trouves.append({
            "chunk_id": chunk_id,
            "titre": doc["title"],
            "section": doc["section"],
            "texte": doc["text"],
            "score": float(score),
        })

        if len(passages_trouves) == k:
            break

    return passages_trouves


# Exemple
for rang, p in enumerate(rechercher(questions[0]["question"]), start=1):
    print(f"{rang}. {p['chunk_id']} | {p['score']:.3f} | {p['titre']} / {p['section']}")

1. gpt_3_001 | 0.869 | GPT-3 / Introduction
2. gpt_4_003 | 0.861 | GPT-4 / Capabilities
3. gpt_3_016 | 0.856 | GPT-3 / GPT-3.5 > GPT-3.5 with browsing
4. gpt_3_018 | 0.855 | GPT-3 / Reception > Applications
5. gpt_3_009 | 0.853 | GPT-3 / Training and capabilities


#### 3.4.2 Prompt et réponses


## 4. Comparaison RAG vs Non-RAG

